In [1]:
import os
import pandas as pd
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import numpy as np

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# name function
str_name = 'poc-step-genxii-lgd-lambda-boto3'

Project: 20231010-gen-xii
Step: 12_step_function


### Hyperparameter df

In [3]:
# make a dictionary of hyperparameters, save as df to s3, so I can re-convert it to dict in the images
dict_hyperparameters = {
    # data sets
    'STR_FILENAME_TRAIN': 'df_train_noleaks_pre.gzip',
    'STR_FILENAME_VALID': 'df_valid_noleaks_pre.gzip', # always use the full data set for the valid model
    'STR_FILENAME_TEST': 'df_test_noleaks_pre.gzip', # always use the full data set for the test model
    # ITERATIONS - define once for consistency
    'INT_N_ITERATIONS': 1000,
    # proportion of iterations used for early stopping
    'PROP_EARLY_STOPPING': 0.10,
    # tuning - 2
    'INT_N_TUNING_JOBS_2': 50, # number of tuning jobs in the second tuning job
    # eval metric
    'STR_EVAL_METRIC': 'RMSE',
}

# make df
df = pd.DataFrame(dict_hyperparameters.items(), columns=['keys','values'])

# save
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/03_pricing_lgd/02_model/02_model/12_step_function/{str_filename}'
df.to_csv(str_uri, index=False)

# show
df

,keys,values
0,STR_FILENAME_TRAIN,df_train_noleaks_pre.gzip
1,STR_FILENAME_VALID,df_valid_noleaks_pre.gzip
2,STR_FILENAME_TEST,df_test_noleaks_pre.gzip
3,INT_N_ITERATIONS,1000
4,PROP_EARLY_STOPPING,0.1
5,INT_N_TUNING_JOBS_2,50
6,STR_EVAL_METRIC,RMSE


### File for dynamic concurrency

In [4]:
%%time

list_int_idx = list(np.arange(0,50)) # 50 jobs
df_tuning = pd.DataFrame({
    'int_idx': list_int_idx,
})
str_filename = 'df_tuning.csv'
str_uri = f's3://{str_project}/ad_hoc/poc_lambda_model/{str_filename}'
df_tuning.to_csv(str_uri, index=False)

# show
df_tuning

CPU times: user 4.98 ms, sys: 352 µs, total: 5.33 ms
Wall time: 27.2 ms


,int_idx
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,7
8,8
9,9


### Write ```definition.json```

In [5]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "GetStartingFeats2",
  "States": {
    "GetStartingFeats2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-starting-feats-2:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "Next": "Map"
    },
    "Map": {
      "Type": "Map",
      "ItemProcessor": {
        "ProcessorConfig": {
          "Mode": "DISTRIBUTED",
          "ExecutionType": "STANDARD"
        },
        "StartAt": "Tuning2",
        "States": {
          "Tuning2": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "OutputPath": "$.Payload",
            "Parameters": {
              "Payload.$": "$",
              "FunctionName": "genxii-lgd-tuning-2"
            },
            "Retry": [
              {
                "ErrorEquals": [
                  "Lambda.ServiceException",
                  "Lambda.AWSLambdaException",
                  "Lambda.SdkClientException",
                  "Lambda.TooManyRequestsException"
                ],
                "IntervalSeconds": 1,
                "MaxAttempts": 3,
                "BackoffRate": 2
              }
            ],
            "End": true
          }
        }
      },
      "Label": "Map",
      "MaxConcurrency": 1000,
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": {
          "InputType": "CSV",
          "CSVHeaderLocation": "FIRST_ROW"
        },
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Key": "ad_hoc/poc_lambda_model/df_tuning.csv"
        }
      },
      "ResultWriter": {
        "Resource": "arn:aws:states:::s3:putObject",
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Prefix": "ad_hoc/poc_lambda_model/tuning_runs"
        }
      },
      "Next": "ConcatTuning2"
    },
    "ConcatTuning2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-concat-tuning-2:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "Next": "Map (1)"
    },
    "Map (1)": {
      "Type": "Map",
      "ItemProcessor": {
        "ProcessorConfig": {
          "Mode": "DISTRIBUTED",
          "ExecutionType": "STANDARD"
        },
        "StartAt": "SensitivityAnalysis2",
        "States": {
          "SensitivityAnalysis2": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "OutputPath": "$.Payload",
            "Parameters": {
              "Payload.$": "$",
              "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-sensitivity-2:$LATEST"
            },
            "Retry": [
              {
                "ErrorEquals": [
                  "Lambda.ServiceException",
                  "Lambda.AWSLambdaException",
                  "Lambda.SdkClientException",
                  "Lambda.TooManyRequestsException"
                ],
                "IntervalSeconds": 1,
                "MaxAttempts": 3,
                "BackoffRate": 2
              }
            ],
            "End": true
          }
        }
      },
      "Label": "Map1",
      "MaxConcurrency": 1000,
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": {
          "InputType": "CSV",
          "CSVHeaderLocation": "FIRST_ROW"
        },
        "Parameters": {
          "Bucket": "20231010-gen-xii",
          "Key": "03_pricing_lgd/02_model/02_model/01_lambda_get_starting_feats/df_cols_in_model.csv"
        }
      },
      "Next": "ConcatSensitivity2"
    },
    "ConcatSensitivity2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-concat-sensitivity:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "Next": "GetNFeats2"
    },
    "GetNFeats2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-get-n-feats:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "Next": "Choice"
    },
    "Choice": {
      "Type": "Choice",
      "Choices": [
        {
          "Not": {
            "Variable": "$",
            "StringMatches": "0"
          },
          "Next": "UpdateFeatDropList2"
        }
      ],
      "Default": "ParallelEval"
    },
    "ParallelEval": {
      "Type": "Parallel",
      "End": true,
      "Branches": [
        {
          "StartAt": "ModelEvalValid2",
          "States": {
            "ModelEvalValid2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::lambda:invoke",
              "OutputPath": "$.Payload",
              "Parameters": {
                "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-eval-valid-2:$LATEST"
              },
              "Retry": [
                {
                  "ErrorEquals": [
                    "Lambda.ServiceException",
                    "Lambda.AWSLambdaException",
                    "Lambda.SdkClientException",
                    "Lambda.TooManyRequestsException"
                  ],
                  "IntervalSeconds": 1,
                  "MaxAttempts": 3,
                  "BackoffRate": 2
                }
              ],
              "End": true
            }
          }
        },
        {
          "StartAt": "ModelEvalTest2",
          "States": {
            "ModelEvalTest2": {
              "Type": "Task",
              "Resource": "arn:aws:states:::lambda:invoke",
              "OutputPath": "$.Payload",
              "Parameters": {
                "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-eval-test-2:$LATEST"
              },
              "Retry": [
                {
                  "ErrorEquals": [
                    "Lambda.ServiceException",
                    "Lambda.AWSLambdaException",
                    "Lambda.SdkClientException",
                    "Lambda.TooManyRequestsException"
                  ],
                  "IntervalSeconds": 1,
                  "MaxAttempts": 3,
                  "BackoffRate": 2
                }
              ],
              "End": true
            }
          }
        }
      ]
    },
    "UpdateFeatDropList2": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-lgd-update-feats:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 1,
          "MaxAttempts": 3,
          "BackoffRate": 2
        }
      ],
      "Next": "GetStartingFeats2"
    }
  }
}

Overwriting definition.json


#### Load and set as string

In [7]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

### Create state machine

In [8]:
cls_client_sfn = boto3.client('stepfunctions')

In [9]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [10]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'step-genxii-ad-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-feat-select-boto3',
 'step-genxii-ad-model-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3',
 'step-genxii-ad-pre-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-pre-boto3',
 'step-genxii-lgd-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-lgd-feat-select-boto3',
 'step-genxii-lgd-model-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-lgd-model-boto3',
 'step-genxii-lgd-pre-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-lgd-pre-boto3',
 'step-genxii-pd-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-ge

In [11]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine poc-step-genxii-lgd-lambda-boto3 does not exist, so it will not be deleted


In [12]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '137',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Wed, 08 Nov 2023 21:48:32 GMT',
                                      'x-amzn-requestid': '7164de13-eadd-4b12-b8f2-abe454c39e3e'},
                      'HTTPStatusCode': 200,
                      'RequestId': '7164de13-eadd-4b12-b8f2-abe454c39e3e',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2023, 11, 8, 21, 48, 32, 246000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3'}


### Describe state machine

In [13]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2252',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Wed, 08 Nov 2023 21:48:32 GMT',
                                      'x-amzn-requestid': 'b950f188-a83b-4b98-96f2-880f296f345d'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'b950f188-a83b-4b98-96f2-880f296f345d',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2023, 11, 8, 21, 48, 32, 246000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"Lambda Invoke", "States": {"Lambda Invoke": {"Type": "Task", '
               '"Resource": "arn:aws:states:::lambda:invoke", "OutputPath": '
               '"$.

### Execute step function workflow

In [14]:
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )
# pprint(dict_response)

### Clean-up

In [15]:
os.remove('./definition.json')